# 02 – Model Training and Experiments
**Deliverables covered:** #5 Baseline and proposed method, #6 Experiments.

Five configurations, changing **one thing at a time** (split, seed, preprocessing and CV folds are identical):

| ID | Model | What changes |
|---|---|---|
| E1 | Logistic Regression | **Baseline** (default C=1) |
| E2 | Logistic Regression | + `class_weight='balanced'` (imbalance handling) |
| E3 | Random Forest | Default-ish: 300 trees, no depth limit |
| E4 | Random Forest | + `class_weight='balanced'`, `min_samples_leaf=5` (regularised) |
| E5 | Random Forest | Hyper-parameters tuned by `RandomizedSearchCV` (5-fold, scored by ROC-AUC) |

Cross-validation is on the **training set only**; the test set is untouched until notebook 03.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.utils import *
sns.set_theme(style="whitegrid"); np.random.seed(SEED)

import time, joblib, json
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
X_train, X_test, y_train, y_test = load_split()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [2]:
def lr(**kw):
    return Pipeline([("prep", build_preprocessor(X_train, scale=True)),
                     ("clf", LogisticRegression(max_iter=2000, random_state=SEED, **kw))])
def rf(**kw):
    return Pipeline([("prep", build_preprocessor(X_train, scale=False)),
                     ("clf", RandomForestClassifier(random_state=SEED, n_jobs=-1, **kw))])

models = {
    "E1_LR_baseline":      (lr(),                                        {"C":1.0,"class_weight":None}),
    "E2_LR_balanced":      (lr(class_weight="balanced"),                 {"C":1.0,"class_weight":"balanced"}),
    "E3_RF_default":       (rf(n_estimators=300),                        {"n_estimators":300,"max_depth":None,"min_samples_leaf":1,"class_weight":None}),
    "E4_RF_balanced_reg":  (rf(n_estimators=300, class_weight="balanced", min_samples_leaf=5),
                                                                          {"n_estimators":300,"max_depth":None,"min_samples_leaf":5,"class_weight":"balanced"}),
}

In [3]:
scoring = {"roc_auc":"roc_auc","pr_auc":"average_precision","f1":"f1","precision":"precision","recall":"recall"}
rows, fitted = [], {}
for name,(pipe,params) in models.items():
    t=time.time()
    cvres = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    pipe.fit(X_train, y_train); fitted[name]=pipe
    rows.append({"experiment":name, "params":json.dumps(params), **{f"cv_{k}":cvres[f"test_{k}"].mean() for k in scoring},
                 "cv_roc_auc_std":cvres["test_roc_auc"].std(), "fit_seconds":round(time.time()-t,1)})
    print(name, "done")

E1_LR_baseline done
E2_LR_balanced done
E3_RF_default done
E4_RF_balanced_reg done


### E5 – tuned Random Forest

In [4]:
param_dist = {"clf__n_estimators":[200,300,500], "clf__max_depth":[4,6,8,12,None],
              "clf__min_samples_leaf":[1,3,5,10,20], "clf__max_features":["sqrt",0.3,0.5],
              "clf__class_weight":[None,"balanced"]}
t=time.time()
search = RandomizedSearchCV(rf(), param_dist, n_iter=15, cv=cv, scoring="roc_auc", random_state=SEED, n_jobs=1, refit=True)
search.fit(X_train, y_train)
best = search.best_estimator_; fitted["E5_RF_tuned"]=best
print("Best params:", search.best_params_); print("Best CV ROC-AUC:", round(search.best_score_,4))
cvres = cross_validate(best, X_train, y_train, cv=cv, scoring=scoring)
rows.append({"experiment":"E5_RF_tuned","params":json.dumps({k.replace("clf__",""):v for k,v in search.best_params_.items()}),
             **{f"cv_{k}":cvres[f"test_{k}"].mean() for k in scoring}, "cv_roc_auc_std":cvres["test_roc_auc"].std(),
             "fit_seconds":round(time.time()-t,1)})

Best params: {'clf__n_estimators': 300, 'clf__min_samples_leaf': 10, 'clf__max_features': 0.3, 'clf__max_depth': 6, 'clf__class_weight': None}
Best CV ROC-AUC: 0.8487


In [5]:
results = pd.DataFrame(rows).set_index("experiment")
results.round(4).to_csv(RESULTS_DIR/"experiments_cv.csv")
results.drop(columns="params").round(4)

,cv_roc_auc,cv_pr_auc,cv_f1,cv_precision,cv_recall,cv_roc_auc_std,fit_seconds
experiment,,,,,,,
E1_LR_baseline,0.8462,0.6615,0.5929,0.6533,0.5438,0.0126,1.9
E2_LR_balanced,0.8459,0.6600,0.6281,0.5167,0.8013,0.0124,2.2
E3_RF_default,0.8224,0.6147,0.5419,0.6259,0.4783,0.0113,19.9
E4_RF_balanced_reg,0.8453,0.6562,0.6327,0.5383,0.7679,0.0096,14.6
E5_RF_tuned,0.8487,0.6650,0.5724,0.6762,0.4970,0.0102,177.8


In [6]:
for name,pipe in fitted.items(): joblib.dump(pipe, MODELS_DIR/f"{name}.joblib")
print("Saved:", sorted(p.name for p in MODELS_DIR.glob("*.joblib")))

Saved: ['E1_LR_baseline.joblib', 'E2_LR_balanced.joblib', 'E3_RF_default.joblib', 'E4_RF_balanced_reg.joblib', 'E5_RF_tuned.joblib']


**Reading the table:** compare E1 vs E2 (effect of class weights on recall/precision trade-off), E3 vs E4 (effect of regularising a Random Forest), and E4 vs E5 (effect of tuning). Note any run where recall rises while precision falls – this is the usual cost of rebalancing. Write your own interpretation in the final report.